# Øvingsoppgaver 1 – Dyp læring / nevrale nettverk
**Løsningsforslag**  
Data Mining & Analytics – Samling 3, dag 1

Bygge og sammenligne en klassisk modell, et MLP og et CNN og undersøke overtilpasning og effekt av regularisering.

Datasettene er valgt slik at du kan kjøre på en vanlig bærbar PC (CPU) på noen få minutter per oppgave.

*Tallene i tolkningene er fra en kjøring i oktober 2026 (TensorFlow 2.13, CPU) og kan variere litt fra kjøring til kjøring.*

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
rng = np.random.default_rng(42)
tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__)

In [ ]:
klasser = ["T-skjorte/topp", "Bukse", "Genser", "Kjole", "Kåpe",
           "Sandal", "Skjorte", "Joggesko", "Veske", "Ankelstøvel"]

### Oppgave 1 – Data og baseline
- a) Last inn Fashion-MNIST med `tf.keras.datasets.fashion_mnist.load_data()`. Skaler pikselverdiene til til verdier fra 0 til 1 og legg til en kanaldimensjon (`(n, 28, 28, 1)`). Bruk de første 20 000 treningsbildene som treningssett og de siste 5 000 som valideringssett.
- b) Plott tre eksempler fra hver klasse for å visuelt verifisere at bildene stemmer med klassene. Plott et stolpediagram som viser klasse-fordelingen. Er klassene balanserte i treningssettet?
- c) Lag en baseline-modell: logistisk regresjon (scikit-learn) på de flate pikselverdiene (28x28 --> flat ut til 784).

Hva blir nøyakighten (accuracy) og F1 score på testsettet?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

print("Oppgave 1a)")
(X_full, y_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_full = np.expand_dims((X_full / 255.0).astype("float32"), axis=-1)
X_test = np.expand_dims((X_test / 255.0).astype("float32"), axis=-1)
X_train, y_train = X_full[:20_000], y_full[:20_000]
X_val, y_val = X_full[-5_000:], y_full[-5_000:]
print("Trening:", X_train.shape, " validering:", X_val.shape, " test:", X_test.shape)
print("--------------------------------")


print("Oppgave 1b)")
fig, ax = plt.subplots(3, 10, figsize=(16, 5))
for k in range(10):
    for r, i in enumerate(np.where(y_train == k)[0][:3]):
        ax[r, k].imshow(X_train[i, :, :, 0], cmap="gray_r")
        ax[r, k].axis("off")
    ax[0, k].set_title(klasser[k])
plt.show()

fordeling = pd.Series(y_train).map(dict(enumerate(klasser))).value_counts().reindex(klasser)
fordeling.plot.bar(color="C0")
plt.ylabel("Antall bilder")
plt.title("Klassefordeling i treningssettet")
plt.xticks(rotation=45, ha="right")
plt.show()

print(fordeling)
print("--------------------------------")


print("Oppgave 1c)")

logreg = LogisticRegression(max_iter=300).fit(X_train.reshape(len(X_train), -1), y_train)
pred_lr = logreg.predict(X_test.reshape(len(X_test), -1))
acc_lr = accuracy_score(y_test, pred_lr)
f1_lr = f1_score(y_test, pred_lr, average="macro")
print(f"Logistisk regresjon – testnøyaktighet: {acc_lr:.3f}, makro-F1: {f1_lr:.3f}")

**Tolkning:** Klassene er tilnærmet balanserte med 1 935 (*T-skjorte/topp*) til 2 068 (*Skjorte*) bilder per klasse i treningssettet, så nøyaktighet (accuracy) er en rimelig metrikk å bruke. Makro-F1 blir derfor (pga balanse) nesten lik accuracy. Logistisk regresjon på de 784 pikselverdiene gir 0,836 i accuracy og 0,835 i makro-F1.

### Oppgave 2 – Flerlags perseptron (MLP)
- a) Bygg et MLP: `Flatten` → `Dense(256, relu)` → `Dropout(0.3)` → `Dense(128, relu)` → `Dense(10, softmax)`. Hvor mange parametere har modellen? Regn ut antallet for det første `Dense`-laget for hånd.
- b) Tren med `Adam`, `sparse_categorical_crossentropy` og `EarlyStopping(patience=3, restore_best_weights=True)`, maks 20 epoker, batch-størrelse 64. Plott trenings- og validerings-loss.
- c) Hva blir nøyakighten (accuracy) og F1 score på testsettet? Er MLP-modellen bedre enn baseline?

In [ ]:
import time

print("Oppgave 2a)")
mlp = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
], name="MLP")
mlp.summary()
print("Regn ut antall parametere i første Dense-lag for hånd: (784 + 1) · 256 =", (784 + 1) * 256)
print("--------------------------------")


print("Oppgave 2b)")
mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stopp = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
h_mlp = mlp.fit(X_train, y_train, epochs=20, batch_size=64, validation_data=(X_val, y_val), callbacks=[stopp], verbose=0)


plt.plot(h_mlp.history["loss"], label="trening")
plt.plot(h_mlp.history["val_loss"], label="validering")
plt.xlabel("Epoke") 
plt.ylabel("Loss")
plt.legend()
plt.title("MLP")
plt.show()
print("--------------------------------")


print("Oppgave 2c)")
pred_mlp = mlp.predict(X_test, verbose=0).argmax(axis=1)
acc_mlp, f1_mlp = accuracy_score(y_test, pred_mlp), f1_score(y_test, pred_mlp, average="macro")
print(f"MLP: testnøyaktighet: {acc_mlp:.3f}, makro-F1: {f1_mlp:.3f}  ({len(h_mlp.history['loss'])} epoker)")
print(f"Baseline: testnøyaktighet: {acc_lr:.3f}, makro-F1: {f1_lr:.3f}")

**Tolkning:** MLP-modellen har 235 146 parametere, hvorav 200 960 i første `Dense`-lag: (784 + 1) · 256. Hvert piksel er koblet til hvert nevron, pluss en bias-verdi per nevron. Early stopping stoppet treningen etter 10 epoker. Modellen gir 0,862 i accuracy og 0,863 i makro-F1 på testsettet, altså bare ~2,5 prosentpoeng bedre enn baseline. 

### Oppgave 3 – Konvolusjonsnettverk (CNN)
- a) Bygg et CNN: `Conv2D(32, 3, relu, padding="same")` → `MaxPooling2D(2)` → `Conv2D(64, 3, relu, padding="same")` → `MaxPooling2D(2)` → `Flatten` → `Dropout(0.3)` → `Dense(128, relu)` → `Dense(10, softmax)`. Regn ut utdata-dimensjon og antall parametere for hvert lag for hånd, og sjekk mot `model.summary()`.
- b) Tren på samme måte som MLP-modellen og sammenlign testnøyaktighet, antall parametere og treningstid.
- c) Lag en forvirringsmatrise og en tabell med sensitivitet (recall) per klasse. Hvilke klasser forveksles oftest, og hvorfor?

In [ ]:
print("Oppgave 3a)")
cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),   # (28, 28, 32): (3·3·1 + 1)·32 = 320
    tf.keras.layers.MaxPooling2D(2),                                     # (14, 14, 32): 0
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),   # (14, 14, 64): (3·3·32 + 1)·64 = 18 496
    tf.keras.layers.MaxPooling2D(2),                                     # (7, 7, 64): 0
    tf.keras.layers.Flatten(),                                           # (3136,): 0
    tf.keras.layers.Dropout(0.3),                                        # (3136,): 0
    tf.keras.layers.Dense(128, activation="relu"),                       # (128,): 3136·128 + 128 = 401 536
    tf.keras.layers.Dense(10, activation="softmax"),                     # (10,): 128·10 + 10 = 1 290
], name="CNN")
cnn.summary()
print("--------------------------------")

print("Oppgave 3b)")
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stopp = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

h_cnn = cnn.fit(X_train, y_train, epochs=20, batch_size=64, validation_data=(X_val, y_val), callbacks=[stopp], verbose=0)
pred_cnn = cnn.predict(X_test, verbose=0).argmax(axis=1)
acc_cnn = accuracy_score(y_test, pred_cnn)
f1_cnn = f1_score(y_test, pred_cnn, average="macro")

print(pd.DataFrame({
    "testnøyaktighet": [acc_lr, acc_mlp, acc_cnn],
    "makro-F1": [f1_lr, f1_mlp, f1_cnn],
    "parametere": [logreg.coef_.size + logreg.intercept_.size, mlp.count_params(), cnn.count_params()],
}, index=["Logistisk regresjon", "MLP", "CNN"]).round(3))
print("--------------------------------")

print("Oppgave 3b)")
from sklearn.metrics import ConfusionMatrixDisplay, recall_score
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(y_test, pred_cnn, display_labels=klasser, cmap="Blues",
                                        colorbar=False, xticks_rotation=60, ax=ax)
ax.grid(False)
plt.show()
pd.Series(recall_score(y_test, pred_cnn, average=None), index=klasser, name="sensitivitet").sort_values().round(3)

**Tolkning:** CNN-modellen gir 0,898 i både nøyaktighet og makro-F1 og er dermed klart best av de tre modellene (baseline 0,836, MLP 0,862). Den har 421 642 parametere, men nesten alle (401 536) sitter i `Dense`-laget etter `Flatten`. De to konvolusjonslagene har til sammen bare 18 816, fordi samme lille filteret brukes over hele bildet.

*Skjorte* har klart lavest sensitivitet (0,71), fulgt av *Genser* (0,84), *Kåpe* (0,85) og *T-skjorte/topp* (0,85). Forvirringsmatrisen viser at feilene går mellom overdeler: *Skjorte* og *T-skjorte/topp* forveksles oftest (122 og 102 bilder hver vei), og *Skjorte* forveksles også med *Kåpe* og *Genser*, mens *Genser* og *Kåpe* forveksles med hverandre. I 28 × 28 gråtoner skiller disse plaggene seg bare på små detaljer som krage, knapper og ermelengde. Blant skotøyet forveksles *Ankelstøvel* og *Sandal* med *Joggesko*. *Bukse* og *Veske* har tydelige former og sensitivitet over 0,97. Totalnøyaktigheten skjuler altså at modellen er mye dårligere på enkelte klasser.

### Oppgave 4 – Overtilpasning med lite data
- a) Tren et nytt CNN (samme arkitektur, men uten `Dropout`) på bare de første 1 000 treningsbildene i 30 epoker, uten early stopping. Plott læringskurvene.
- b) Tren en ny versjon med dataaugmentering (`RandomFlip("horizontal")`, `RandomRotation(0.05)`, `RandomZoom(0.1)`) først i modellen, `Dropout(0.3)` og early stopping (patience 5). Sammenlign valideringsnøyaktighet og kurver.
- c) Hjelper augmentering? I så fall hvorfor? Er alle typer augmentering fornuftige/plausible for dette datasettet?

In [ ]:
from tensorflow.keras import layers

def lite_cnn(augmentering=None, dropout=0.0):
    model = tf.keras.Sequential()
    model.add(tf.keras.Input(shape=(28, 28, 1)))

    if augmentering is not None:
        model.add(augmentering)

    model.add(layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"))
    model.add(layers.MaxPooling2D(pool_size=2))

    model.add(layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"))
    model.add(layers.MaxPooling2D(pool_size=2))

    model.add(layers.Flatten())

    if dropout > 0:
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dense(10, activation="softmax")) 

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


X_lite, y_lite = X_train[:1_000], y_train[:1_000]

print("Oppgave 4a)")
uten_aug = lite_cnn()
h_uten = uten_aug.fit(X_lite, y_lite, epochs=30, batch_size=32, validation_data=(X_val, y_val), verbose=0)
print("--------------------------------")

print("Oppgave 4b)")
aug = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomRotation(0.05),
                           tf.keras.layers.RandomZoom(0.1)])
med_aug = lite_cnn(augmentering=aug, dropout=0.3)
h_med = med_aug.fit(X_lite, y_lite, epochs=60, batch_size=32, validation_data=(X_val, y_val), verbose=0,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for a, h, t in [(ax[0], h_uten, "Uten regularisering"), (ax[1], h_med, "Augmentering + dropout + early stopping")]:
    a.plot(h.history["loss"], label="trening"); a.plot(h.history["val_loss"], label="validering")
    a.set_title(t); a.set_xlabel("Epoke"); a.legend()
ax[0].set_ylabel("Loss")
plt.show()
print(f"Valideringsnøyaktighet – uten: {uten_aug.evaluate(X_val, y_val, verbose=0)[1]:.3f}, "
      f"med: {med_aug.evaluate(X_val, y_val, verbose=0)[1]:.3f} ({len(h_med.history['loss'])} epoker)")

**Tolkning:** Uten regularisering går treningstapet mot null, mens valideringstapet når et minimum etter rundt 8 epoker og så stiger til ca. 0,83. Det ser ut til at modellen overtilpasser seg det lille treningssettet (1 000 bilder). Med augmentering, dropout og early stopping følger trenings- og valideringskurvene hverandre tett, og valideringstapet ender lavere (ca. 0.57). Vi har motvirket overtilpasningen.

Augmenteringen må bevare klassen. Horisontal speiling, små rotasjoner og zoom er plausible for klær, men vertikal speiling gir opp-ned sko og kjoler som sannynligvis ikke forekommer i testdata. For andre data, f.eks. sifrene 6 og 9 eller røntgenbilder med venstre/høyre-side kan speiling endre betydningen.

### Oppgave 5 – Refleksjon
Når vil du velge dyp læring/nevrale nettverk framfor metodene fra samling 2 (logistisk regresjon, random forest osv.)? Se om kan komme på tre faktorer. 

**Tolkning:** Noen eksempler på faktorer:
- **Datatype:** bilder, lyd, tekst og signaler (f.eks. EKG) har lokale mønstre som CNN klarer å utnytte. På tabelldata er trebaserte modeller og logistisk regresjon ofte like gode.
- **Datamengde:** dype nettverk trenger mye data, med mindre vi kan bruke transfer learning.
- **Tolkbarhet:** logistisk regresjon har tolkbare koeffisienter, nevrale nettverk krever egne forklaringsmetoder.
- **Regnekraft og vedlikehold:** trening, GPU-er og hyperparametersøk koster tid og penger.
- **Robusthet:** dype nettverk er sårbare for adversarial angrep og distribusjonsskift.

Alltid lurt å starte med en enkel baseline, slik som i oppgave 1.